In [78]:
spark

In [79]:
%run /opt/spark-notebooks/silver_base/parsed_data.ipynb

Master DataFrame 'parsed_df' successfully created and listening to Bronze!


In [80]:
from pyspark.sql.functions import col, explode
# 1. Create the database table mapping
spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_base.order_event_fact_RT (
        order_id STRING,
        event_type STRING,
        event_time TIMESTAMP,
        source_system STRING,
        bronze_ingest_ts TIMESTAMP
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/base/order_event_fact_RT'
""")

DataFrame[]

In [81]:
events_exploded_df = parsed_df.select(
    col("data.order.order_id").alias("order_id"),
    explode(col("data.order.order_events")).alias("event"),
    col("source_system"),
    col("bronze_ingest_ts")
)


In [82]:
silver_order_event_fact_df = events_exploded_df.select(
    col("order_id"),
    col("event.event_type").alias("event_type"),
    col("event.event_time").cast("timestamp").alias("event_time"),
    col("source_system"),
    col("bronze_ingest_ts")
)

In [83]:
event_fact_query = (
    silver_order_event_fact_df.writeStream
    .format("delta")
    .queryName("silver_order_event_fact_stream")
    .outputMode("append")
    .option("checkpointLocation", "/opt/spark-data/checkpoints/silver/base/order_event_fact_RT/v1")
    .trigger(processingTime="30 seconds")
    .start("/opt/spark-data/delta/silver/base/order_event_fact_RT")
)
print("Started silver_order_event_fact_RT continuous stream! 🚀")

Started silver_order_event_fact_RT continuous stream! 🚀


In [85]:
spark.sql("SELECT * FROM silver_base.order_event_fact_RT").limit(20).toPandas()


,order_id,event_type,event_time,source_system,bronze_ingest_ts
0,ORD-10001,CREATED,2026-01-03 09:05:12,order_service,2026-03-27 16:27:10.891
1,ORD-10001,PAID,2026-01-03 09:06:10,order_service,2026-03-27 16:27:10.891
2,ORD-10001,SHIPPED,2026-01-03 09:45:00,order_service,2026-03-27 16:27:10.891
3,ORD-10002,CREATED,2026-01-03 09:20:00,order_service,2026-03-27 16:27:10.891
4,ORD-10002,PAID,2026-01-03 09:22:30,order_service,2026-03-27 16:27:10.891
5,ORD-10003,CREATED,2026-01-03 09:40:00,order_service,2026-03-27 16:27:10.891
6,ORD-10003,PAID,2026-01-03 09:42:00,order_service,2026-03-27 16:27:10.891
7,ORD-10003,SHIPPED,2026-01-03 10:15:00,order_service,2026-03-27 16:27:10.891
8,ORD-10004,CREATED,2026-01-03 10:00:00,order_service,2026-03-27 16:27:10.891
9,ORD-10005,CREATED,2026-01-03 10:10:00,order_service,2026-03-27 16:27:10.891
